# SQL manager notebook

This notebook is a **SQL-only** sandbox for `boti_data.db.sql_manager` and nearby SQL helpers. It exercises:

- sync SQL resource creation
- reflected model creation with `SqlAlchemyModelBuilder`
- raw SQL and ORM-style `SELECT` queries
- simple filter patterns (`=`, `LIKE`, `IN`)
- query-only behavior
- async SQL resource usage when an async driver is available

In [1]:
import os
import sys
import tempfile
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src" / "boti").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()

SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

import pandas as pd
from sqlalchemy import create_engine, inspect, select, text

from boti_data.db import ensure_greenlet_available
from boti_data.db.sql_manager import AsyncSqlDatabaseResource, SqlDatabaseConfig, SqlDatabaseResource
from boti_data.db.sql_model_builder import SqlAlchemyModelBuilder

print(PROJECT_ROOT)

/Users/lvalverdeb/TeamDev/repo-split/boti-data


## Create a temporary SQL sandbox

In [2]:
tmp_root = Path(tempfile.mkdtemp(prefix="boti_sql_manager_"))
db_path = tmp_root / "sql_manager.db"

people = pd.DataFrame(
    {
        "id": [1, 2, 3, 4],
        "name": ["Alice", "Bob", "Carla", "Diego"],
        "status": ["active", "inactive", "active", "active"],
        "team": ["analytics", "platform", "analytics", "ops"],
        "age": [31, 28, 35, 42],
    }
)

orders = pd.DataFrame(
    {
        "id": [100, 101, 102, 103],
        "person_id": [1, 1, 3, 4],
        "amount": [120.50, 49.99, 250.00, 90.00],
        "priority": ["high", "low", "high", "medium"],
    }
)

engine = create_engine(f"sqlite:///{db_path}")
with engine.begin() as conn:
    people.to_sql("people", conn, index=False, if_exists="replace")
    orders.to_sql("orders", conn, index=False, if_exists="replace")
engine.dispose()

sync_dsn = f"sqlite:///{db_path}"
readonly_dsn = f"sqlite:///file:{db_path}?mode=ro&uri=true"

print({"tmp_root": str(tmp_root), "sync_dsn": sync_dsn, "readonly_dsn": readonly_dsn})

{'tmp_root': '/var/folders/j1/c0fy94996q51rf0nkcvg577m0000gn/T/boti_sql_manager_co3dd5jk', 'sync_dsn': 'sqlite:////var/folders/j1/c0fy94996q51rf0nkcvg577m0000gn/T/boti_sql_manager_co3dd5jk/sql_manager.db', 'readonly_dsn': 'sqlite:///file:/var/folders/j1/c0fy94996q51rf0nkcvg577m0000gn/T/boti_sql_manager_co3dd5jk/sql_manager.db?mode=ro&uri=true'}


## Sync resource creation and reflected model building

In [3]:
sync_config = SqlDatabaseConfig(
    connection_url=sync_dsn,
    poolclass="sqlalchemy.pool.NullPool",
    query_only=False,
)

with SqlDatabaseResource(sync_config) as db:
    inspector = inspect(db.engine)
    tables = inspector.get_table_names()
    people_model = SqlAlchemyModelBuilder(db.engine, "people").build_model()
    orders_model = SqlAlchemyModelBuilder(db.engine, "orders").build_model()
    
    model_summary = {
        "tables": tables,
        "people_columns": list(people_model.__table__.columns.keys()),
        "orders_columns": list(orders_model.__table__.columns.keys()),
    }

model_summary

[2026-06-12 11:45:37][WARNING][SqlModelRegistry] Missing native Primary Key on people. Synthesizing mapper bindings using primary sequential column fallback. Ensure ORM mutation sequences validate bounds manually.
[2026-06-12 11:45:37][WARNING][SqlModelRegistry] Missing native Primary Key on orders. Synthesizing mapper bindings using primary sequential column fallback. Ensure ORM mutation sequences validate bounds manually.


{'tables': ['orders', 'people'],
 'people_columns': ['id', 'name', 'status', 'team', 'age'],
 'orders_columns': ['id', 'person_id', 'amount', 'priority']}

## Raw SQL `SELECT` queries

In [4]:
with SqlDatabaseResource(sync_config) as db:
    with db.engine.connect() as conn:
        all_people = conn.execute(
            text("SELECT id, name, status, team, age FROM people ORDER BY id")
        ).mappings().all()
        active_count = conn.execute(
            text("SELECT COUNT(*) FROM people WHERE status = :status"),
            {"status": "active"},
        ).scalar_one()

{
    "all_people": [dict(row) for row in all_people],
    "active_count": active_count,
}

{'all_people': [{'id': 1,
   'name': 'Alice',
   'status': 'active',
   'team': 'analytics',
   'age': 31},
  {'id': 2,
   'name': 'Bob',
   'status': 'inactive',
   'team': 'platform',
   'age': 28},
  {'id': 3,
   'name': 'Carla',
   'status': 'active',
   'team': 'analytics',
   'age': 35},
  {'id': 4, 'name': 'Diego', 'status': 'active', 'team': 'ops', 'age': 42}],
 'active_count': 3}

## ORM-style `SELECT` queries and filters

In [5]:
with SqlDatabaseResource(sync_config) as db:
    People = SqlAlchemyModelBuilder(db.engine, "people").build_model()
    Orders = SqlAlchemyModelBuilder(db.engine, "orders").build_model()
    
    with db.session() as session:
        active_people = session.execute(
            select(People).where(People.status == "active").order_by(People.id)
        ).scalars().all()
        analytics_or_ops = session.execute(
            select(People).where(People.team.in_(["analytics", "ops"])).order_by(People.id)
        ).scalars().all()
        name_like_a = session.execute(
            select(People).where(People.name.like("A%")).order_by(People.id)
        ).scalars().all()
        joined_orders = session.execute(
            select(People.name, Orders.amount, Orders.priority)
            .join(Orders, Orders.person_id == People.id)
            .where(Orders.priority.in_(["high", "medium"]))
            .order_by(Orders.id)
        ).all()

{
    "active_people": [row.name for row in active_people],
    "analytics_or_ops": [row.name for row in analytics_or_ops],
    "name_like_a": [row.name for row in name_like_a],
    "joined_orders": [tuple(row) for row in joined_orders],
}

{'active_people': ['Alice', 'Carla', 'Diego'],
 'analytics_or_ops': ['Alice', 'Carla', 'Diego'],
 'name_like_a': ['Alice'],
 'joined_orders': [('Alice', 120.5, 'high'),
  ('Carla', 250.0, 'high'),
  ('Diego', 90.0, 'medium')]}

## Query-only mode

In [6]:
readonly_config = SqlDatabaseConfig(
    connection_url=readonly_dsn,
    poolclass="sqlalchemy.pool.NullPool",
    query_only=True,
)

readonly_summary = {}
with SqlDatabaseResource(readonly_config) as db:
    with db.engine.connect() as conn:
        readonly_summary["selected_name"] = conn.execute(
            text("SELECT name FROM people WHERE id = :id"),
            {"id": 1},
        ).scalar_one()

    try:
        with db.session() as session:
            session.commit()
    except Exception as exc:
        readonly_summary["commit_blocked"] = type(exc).__name__
        readonly_summary["commit_message"] = str(exc)

readonly_summary

{'selected_name': 'Alice',
 'commit_blocked': 'SQLAlchemyError',
 'commit_message': 'SqlDatabaseResource sessions are read-only.'}

## Async resource setup

The async section works in either of these modes:

1. self-contained SQLite via `aiosqlite` if it is installed
2. a user-supplied async DSN via `BOTI_SQL_NOTEBOOK_ASYNC_DSN`

If neither is available, the async cells will print a skip message.

In [7]:
async_dsn = os.environ.get("BOTI_SQL_NOTEBOOK_ASYNC_DSN")
async_source = None

if async_dsn:
    async_source = "env"
else:
    try:
        import aiosqlite  # noqa: F401
    except ImportError:
        async_dsn = None
    else:
        async_dsn = f"sqlite+aiosqlite:///{db_path}"
        async_source = "local_aiosqlite"

{
    "async_dsn": async_dsn,
    "async_source": async_source,
    "message": (
        "Ready for async checks"
        if async_dsn is not None
        else "Set BOTI_SQL_NOTEBOOK_ASYNC_DSN or install aiosqlite to run the async cells."
    ),
}

{'async_dsn': None,
 'async_source': None,
 'message': 'Set BOTI_SQL_NOTEBOOK_ASYNC_DSN or install aiosqlite to run the async cells.'}

In [8]:
async def run_async_sql_checks(dsn: str) -> dict[str, object]:
    ensure_greenlet_available()
    config = SqlDatabaseConfig(
        connection_url=dsn,
        poolclass="sqlalchemy.pool.NullPool",
        query_only=False,
    )

    async with AsyncSqlDatabaseResource(config) as db:
        People = await SqlAlchemyModelBuilder(db.engine, "people").build_model_async()
        async with db.session() as session:
            active_people = (
                await session.execute(
                    select(People).where(People.status == "active").order_by(People.id)
                )
            ).scalars().all()
            age_filtered = await session.execute(
                text("SELECT COUNT(*) FROM people WHERE age >= :age"),
                {"age": 30},
            )

    return {
        "active_people": [row.name for row in active_people],
        "age_gte_30": age_filtered.scalar_one(),
    }

In [9]:
if async_dsn is None:
    print("Async demo skipped. Provide BOTI_SQL_NOTEBOOK_ASYNC_DSN or install aiosqlite.")
else:
    await run_async_sql_checks(async_dsn)

Async demo skipped. Provide BOTI_SQL_NOTEBOOK_ASYNC_DSN or install aiosqlite.


## Cleanup note

The notebook keeps the temporary SQLite database on disk for the current session under `tmp_root`. Re-run the setup cell whenever you want a fresh sandbox.